# Phase S — SCRUB
## Data Cleaning, Frequency Alignment, Z-scoring & Feature Engineering

**OSEMN Step 2**: Transform raw mixed-frequency data into a clean, aligned, normalised feature matrix.

Key steps:
1. Resample all series → Business Month End (`BME`)
2. Interpolate quarterly GDP (linear, ≤3 month gap)
3. Apply rolling 36-month Z-scores (prevents era drift)
4. Engineer derived features: YoY growth, credit impulse, yield curve flags, FSI

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt

from src.config_loader import load_config
from src.obtain.fred_loader import FREDLoader
from src.obtain.market_loader import MarketLoader
from src.scrub.preprocessor import Preprocessor
from src.scrub.feature_engineer import FeatureEngineer

cfg = load_config('../config/config.yaml')

In [ ]:
# Load raw data (from cache)
fred_data = FREDLoader(cfg).load()
market_data = MarketLoader(cfg).load()
print('Data loaded from cache.')

## 2.1 Preprocessing — Alignment & Normalisation

In [ ]:
preprocessor = Preprocessor(cfg)
df_zscored, df_raw = preprocessor.build_feature_matrix(fred_data)

print(f'Raw monthly:    {df_raw.shape}  ({df_raw.index[0].date()} → {df_raw.index[-1].date()})')
print(f'Z-scored:       {df_zscored.shape}')
print(f'NaN fraction:   {df_zscored.isna().mean().mean():.2%}')
df_raw.head()

## 2.2 Visualise Quarterly Interpolation (GDP)

In [ ]:
if 'gdp_real' in df_raw.columns:
    raw_gdp = fred_data['gdp_real'].resample('BME').last()
    interp_gdp = df_raw['gdp_real']
    
    fig, ax = plt.subplots(figsize=(14, 4))
    interp_gdp.loc['2000':].plot(ax=ax, label='Interpolated (monthly)', color='steelblue')
    raw_gdp.loc['2000':].plot(ax=ax, label='Raw quarterly points', style='o',
                               color='red', markersize=4)
    ax.set_title('Real GDP: Quarterly → Monthly Interpolation')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 2.3 Z-scored Features — Yield Curve & Credit Spread

In [ ]:
key_features = [c for c in ['yield_curve_10y2y', 'credit_spread_baa', 'unemployment']
                if c in df_zscored.columns]

fig, axes = plt.subplots(len(key_features), 1, figsize=(16, 4*len(key_features)), sharex=True)
if len(key_features) == 1:
    axes = [axes]

for ax, col in zip(axes, key_features):
    df_zscored[col].plot(ax=ax, color='steelblue', linewidth=1)
    ax.axhline(0, color='black', lw=0.5, ls='--')
    ax.axhline(2, color='red', lw=0.5, ls=':', alpha=0.7)
    ax.axhline(-2, color='green', lw=0.5, ls=':', alpha=0.7)
    ax.set_title(f'{col} (36-month rolling Z-score)')

plt.tight_layout()
plt.show()

## 2.4 Feature Engineering — Derived Signals

In [ ]:
fe = FeatureEngineer(cfg)
features = fe.engineer(df_raw, df_zscored)

print(f'Full feature matrix: {features.shape}')
print(f'All columns ({len(features.columns)}):')
for i, col in enumerate(features.columns):
    print(f'  {i+1:3d}. {col}')

## 2.5 Financial Stress Index

In [ ]:
fsi_col = cfg['feature_engineering']['stress_index']['name']
if fsi_col in features.columns:
    fig, ax = plt.subplots(figsize=(16, 5))
    features[fsi_col].plot(ax=ax, color='darkred', linewidth=1.2, label='FSI')
    ax.fill_between(features.index, features[fsi_col], 0,
                    where=features[fsi_col]>0, alpha=0.3, color='red', label='Stress')
    ax.fill_between(features.index, features[fsi_col], 0,
                    where=features[fsi_col]<0, alpha=0.3, color='green', label='Calm')
    ax.axhline(0, color='black', lw=0.7)
    ax.set_title('Financial Stress Index (composite Z-score)', fontsize=13)
    ax.legend()
    plt.tight_layout()
    plt.show()

## 2.6 Asset Returns Matrix

In [ ]:
assets = list(cfg['data']['market']['etfs'].keys())
asset_returns = preprocessor.build_returns_matrix(market_data, assets)

print(f'Returns matrix: {asset_returns.shape}')
print(f'\nAnnualised statistics:')
ann = pd.DataFrame({
    'Ann. Return': asset_returns.mean() * 12,
    'Ann. Vol':    asset_returns.std() * (12**0.5),
    'Sharpe':      asset_returns.mean() / asset_returns.std() * (12**0.5)
})
ann.style.format('{:.2%}', subset=['Ann. Return','Ann. Vol']).format('{:.2f}', subset='Sharpe')

## Summary
- All FRED series aligned to monthly Business Month End
- GDP quarterly gap interpolated (linear, ≤3 month)
- Rolling 36-month Z-scores applied to all features
- Derived: YoY growth, credit impulse, yield curve flags, FSI
- **Next**: `03_explore.ipynb` — EDA + BIC state selection